In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
benchmark_name = "eda-speedtests"
from pathlib import Path
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import regex as re # For String searches
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

### cell 0 – optimized for cudf
from pathlib import Path

base_input = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"
mobile_list = []
broadband_list = []

for root, _, files in os.walk(base_input):
    for file in files:
        filepath = Path(root) / file
        # read directly into GPU and convert dtypes to match original (ensures 'Name' is string dtype)
        df = pd.read_csv(filepath, thousands=',').convert_dtypes()
        # rename if needed (cudf.rename is GPU)
        df.rename(columns={'Number of Record':'Number of Records'}, inplace=True)
        # extract year & quarter once per file
        yr = int(re.search(r'year_(\d+)_quarter', file).group(1))
        qt = int(re.search(r'quarter_(\d+)\.csv', file).group(1))
        df['Year'] = yr
        df['Quarter'] = qt
        # collect into lists, defer concat
        if 'mobile' in file:
            mobile_list.append(df)
        else:
            broadband_list.append(df)

# single GPU concat per category instead of per-file
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)

# GPU astype & sort (assignment, not fallback inplace)
Mobile_df = Mobile_df.astype({'Year':'int64','Quarter':'int64'})
Broadband_df = Broadband_df.astype({'Year':'int64','Quarter':'int64'})
Mobile_df = Mobile_df.sort_values(['Year','Quarter'])
Broadband_df = Broadband_df.sort_values(['Year','Quarter'])

# expand datasets with fewer GPU concats
factor = 1000
Mobile_df = pd.concat([Mobile_df] * (factor * 2), ignore_index=True)
Broadband_df = pd.concat([Broadband_df] * factor, ignore_index=True)

Mobile_df.info()
Broadband_df.info()

In [ ]:
### cell 1 ###

unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

In [ ]:
### cell 2 ###

unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

In [ ]:
### cell 3 ###

# Check for missing values
Mobile_df.isna().any()

In [ ]:
### cell 4 ###

Broadband_df.isna().any()

In [ ]:
### cell 5 ###

Mobile_df.shape

In [ ]:
### cell 6 ###

unique_countries_mobile.loc[unique_countries_mobile['Year'] < 10, 'Year']

In [ ]:
### cell 7 ###

unique_countries_broadband[unique_countries_broadband.Year < 10]['Year']

In [ ]:
### cell 8 ###

cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile stats: group and agg first/last, then compute deltas without .xs
mobile_agg = Mobile_df.groupby('Name')[cols].agg(['first', 'last'])
Mobile_Stats = (
    (mobile_agg[('Avg. Avg D Kbps', 'last')] - mobile_agg[('Avg. Avg D Kbps', 'first')])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(mobile_agg[('Avg. Avg U Kbps', 'last')] - mobile_agg[('Avg. Avg U Kbps', 'first')]),
        Change_Latency=(mobile_agg[('Avg Lat Ms',       'last')] - mobile_agg[('Avg Lat Ms',       'first')])
    )
)

# Broadband stats: same pattern
broad_agg = Broadband_df.groupby('Name')[cols].agg(['first', 'last'])
Broadband_Stats = (
    (broad_agg[('Avg. Avg D Kbps', 'last')] - broad_agg[('Avg. Avg D Kbps', 'first')])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(broad_agg[('Avg. Avg U Kbps', 'last')] - broad_agg[('Avg. Avg U Kbps', 'first')]),
        Change_Latency=(broad_agg[('Avg Lat Ms',       'last')] - broad_agg[('Avg Lat Ms',       'first')])
    )
)

# Combine only the download-change columns
Total_Stats = (
    Mobile_Stats['Change_Download'].to_frame('Mobile')
    .join(Broadband_Stats['Change_Download'].to_frame('Broadband'))
)

In [ ]:
### cell 9 ###

# Optimized to use a single GPU‐accelerated between() call
ImprovedCountries_B = Broadband_Stats[
    Broadband_Stats['Change_Download'].between(1, 2999)
]

In [ ]:
### cell 10 ###

ImprovedCountries_B = Broadband_Stats[
    Broadband_Stats['Change_Download'].between(3000, 8000, inclusive='neither')
]

In [ ]:
### cell 11 ###

### cell 11 optimized ###
# Use a string for `inclusive` to exclude both endpoints (i.e. >8000 and <16000)
ImprovedCountries_B = Broadband_Stats[
    Broadband_Stats['Change_Download'].between(8000, 16000, inclusive='neither')
]

In [ ]:
### cell 12 ###

ImprovedCountries_B = Broadband_Stats[
    Broadband_Stats['Change_Download'].between(16000, 60000, inclusive='neither')
]

In [ ]:
### cell 13 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] >= 60000)]

In [ ]:
### cell 14 ###

# Use cudf's GPU-accelerated Series.lt instead of the Python < operator
ImprovedCountries_B = Broadband_Stats[Broadband_Stats['Change_Download'].lt(0)]
Countries = ImprovedCountries_B.index

In [ ]:
### cell 15 ###

Mobile_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 16 ###

Broadband_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 17 ###

# Compute relative changes for Mobile_df
_mb = Mobile_df.groupby('Name')
_mobile_first = _mb.first()[['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']]
_mobile_last  = _mb.last()[['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']]
Mobile_Stats_relative = ((_mobile_last - _mobile_first) / _mobile_first).astype('float64')
Mobile_Stats_relative.columns = ['Change_Download', 'Change_Upload', 'Change_Latency']

# Compute relative changes for Broadband_df
_bb = Broadband_df.groupby('Name')
_broadband_first = _bb.first()[['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']]
_broadband_last  = _bb.last()[['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']]
Broadband_Stats_relative = ((_broadband_last - _broadband_first) / _broadband_first).astype('float64')
Broadband_Stats_relative.columns = ['Change_Download', 'Change_Upload', 'Change_Latency']

In [ ]:
### cell 18 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 2)]

In [ ]:
### cell 19 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 1) & (Broadband_Stats_relative['Change_Download'] < 2)]

In [ ]:
### cell 20 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.5) & (Broadband_Stats_relative['Change_Download'] < 1)]

In [ ]:
### cell 21 ###

# cell 21 optimized
change_download = Broadband_Stats_relative['Change_Download']
mask = (change_download >= 0.2) & (change_download < 0.5)
ImprovedCountries_B = Broadband_Stats_relative[mask]

In [ ]:
### cell 22 ###

ImprovedCountries_B = Broadband_Stats_relative[
    Broadband_Stats_relative['Change_Download'].between(0, 0.2, inclusive='left')
]

In [ ]:
### cell 23 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] < 0)]

In [ ]:
### cell 24 ###

Broadband_Stats_relative.sort_values(by=['Change_Download'])

In [ ]:
### cell 25 ###

Mobile_Stats_relative.sort_values(by=['Change_Download'])